# Piloto de predicción horaria con torres GRU-D

Este notebook reutiliza los módulos del proyecto. La configuración inicial toma **300 pacientes en total del train de A**, conserva todas sus horas y reserva el 20 % de los pacientes para validación. La prueba final está desactivada por defecto.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.


In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATA_DIR = ROOT / "data" / "processed"


## 1. Configuración del piloto

`source` admite `"A"`, `"B"` y `"A+B"`. Para usar todos los pacientes de entrenamiento, cambia `sample_patients` a `None`. Con `folds=None` se usa una sola partición; un entero como `5` activa la validación cruzada.

`trials=1` entrena solo la configuración base. Con `trials=4` se prueban esa configuración y tres alternativas de la búsqueda aleatoria. El costo de la búsqueda aumenta con los ensayos y los folds.


In [2]:
settings = dict(
    source="A",
    sample_patients=None,
    validation_fraction=0.20,
    folds=3,
    trials=1,
    epochs=20,
    patience=4,
    seed=2026,
)

base = Hyperparameters(
    window=15,
    units=16,
    dense_units=32,
    dropout=0.20,
    learning_rate=0.001,
    batch_size=64,
)


## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [3]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(split_patients(
    frame,
    validation_fraction=settings["validation_fraction"],
    folds=settings["folds"],
    seed=settings["seed"],
    sample_patients=settings["sample_patients"],
))

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append({
        "partición": name,
        "pacientes": len(patient_labels),
        "pacientes positivos": int(patient_labels.sum()),
        "horas": len(partition),
    })
display(pd.DataFrame(summary).set_index("partición"))


,pacientes,pacientes positivos,horas
partición,,,
train,10846,955,418590
validation,5423,477,212441


In [4]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print("Horas válidas de la primera ventana:", inputs["valid_hours"][0].astype(int).tolist())
print("Etiqueta de la primera ventana:", int(labels[0, 0]))


Formas de entrada: {'temporal': (54, 15, 26, 4), 'valid_hours': (54, 15), 'static': (54, 10)}
Forma de las etiquetas: (54, 1)
Paciente: A_000001
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0


## 3. Entrenar y revisar los resultados

Cada ejecución de la celda crea una carpeta nueva en `runs/` con el modelo `best.keras`, la configuración, los tiempos y las métricas. Las métricas de validación sirven para elegir el modelo y el umbral. Con una muestra de 300 pacientes, el objetivo del piloto es verificar el flujo y estimar el costo.


In [5]:
run_dir = ROOT / "runs" / f"notebook-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))


Trial 1/1, fold 1/3: Hyperparameters(window=15, units=16, dense_units=32, dropout=0.2, learning_rate=0.001, batch_size=64)



c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/20
6541/6541 - 713s - 109ms/step - loss: 0.0998 - pr_auc: 0.0572 - val_loss: 0.0916 - val_pr_auc: 0.0873
Epoch 2/20
6541/6541 - 665s - 102ms/step - loss: 0.0936 - pr_auc: 0.0885 - val_loss: 0.0907 - val_pr_auc: 0.0962
Epoch 3/20
6541/6541 - 687s - 105ms/step - loss: 0.0913 - pr_auc: 0.1037 - val_loss: 0.0901 - val_pr_auc: 0.0958
Epoch 4/20
6541/6541 - 615s - 94ms/step - loss: 0.0897 - pr_auc: 0.1157 - val_loss: 0.0899 - val_pr_auc: 0.1032
Epoch 5/20
6541/6541 - 1124s - 172ms/step - loss: 0.0883 - pr_auc: 0.1285 - val_loss: 0.0901 - val_pr_auc: 0.1022
Epoch 6/20
6541/6541 - 2907s - 445ms/step - loss: 0.0869 - pr_auc: 0.1405 - val_loss: 0.0929 - val_pr_auc: 0.0936
Epoch 7/20
6541/6541 - 1519s - 232ms/step - loss: 0.0855 - pr_auc: 0.1521 - val_loss: 0.0923 - val_pr_auc: 0.1036
Epoch 8/20
6541/6541 - 1100s - 168ms/step - loss: 0.0840 - pr_auc: 0.1693 - val_loss: 0.0915 - val_pr_auc: 0.0994
Epoch 9/20
6541/6541 - 722s - 110ms/step - loss: 0.0826 - pr_auc: 0.1906 - val_loss: 0.0935 -

c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/20


KeyboardInterrupt: 

In [11]:
display(pd.read_csv(run_dir / "trials.csv").round(4))

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial", "fold", "train_windows", "validation_windows",
    "preparation_seconds", "epochs_run", "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


,trial,window,units,dense_units,dropout,learning_rate,batch_size,mean_validation_ap,std_validation_ap
0,1,15,8,32,0.2,0.0010,128,0.1115,0.0
1,2,8,32,32,0.2,0.0003,128,0.1059,0.0
2,3,15,32,64,0.1,0.0010,256,0.0886,0.0
3,4,8,8,64,0.2,0.0003,128,0.1087,0.0
4,5,24,32,64,0.2,0.0010,64,0.0904,0.0
5,6,15,16,32,0.2,0.0010,64,0.1128,0.0
6,7,15,16,32,0.1,0.0003,128,0.0902,0.0
7,8,15,8,64,0.1,0.0010,128,0.0956,0.0
8,9,8,16,32,0.2,0.0003,128,0.1026,0.0
9,10,15,8,32,0.1,0.0003,128,0.0867,0.0


,trial,fold,train_windows,validation_windows,preparation_seconds,epochs_run,seconds_per_epoch
0,1,1,62270,15554,1.92,17,52.25
2,2,1,62270,15554,1.73,20,77.12
4,3,1,62270,15554,0.76,20,115.40
6,4,1,62270,15554,0.63,20,20.71
8,5,1,62270,15554,0.74,17,173.03
10,6,1,62270,15554,0.84,14,61.89
12,7,1,62270,15554,0.82,20,54.91
14,8,1,62270,15554,0.67,16,34.99
16,9,1,62270,15554,0.80,20,34.32
18,10,1,62270,15554,0.57,20,39.02


## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [7]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(run_dir / f"test_{test_source}_predictions.csv", index=False)
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))
